In [1]:
#Amo demo
#FM 15.12.2025
#Amo evaluation
#FM 28.11.2025
import sys
sys.path.append('amos') 
# Import all ML orchestration functions
from amos.ml_orchestration import (
    ml_exp_with_timing,  # Main experiment function with timing preservation
    extract_timing_structure,
    save_midi_with_exact_timing_structure,
    KerasClassifierWrapper,
    build_lstm_classifier,
    build_transformer_classifier,
    clf_predict,
    defineXy,           # Data preparation function
    split_and_encode    # Train/test split with encoding
)
# Import MIDI processing functions  
from amos.midi2df2midi import midi_to_dataframe, save_midi_from_df
from amos.mappings import fill_quaterna_columns, learn_quaterna_mapping
# Import amo funtion
from amos.ml_orchestration import amo_with_doublings_multiclass

import pandas as pd
import numpy as np
from collections import defaultdict


from xgboost import XGBClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier

import matplotlib.pyplot as plt



/home/francesco/anaconda3/envs/auto-orch/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
2025-12-16 23:43:20.092504: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765925000.134116  294473 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been

In [2]:
def transpose(note, inverse=False, n_semitones=12):
    # Example: transpose pitch
    if inverse:
        n_semitones = - n_semitones
    new_note = note.copy()
    new_note['pitch'] = note['pitch'] + n_semitones
    return [ new_note ]

In [3]:
transformations = [
    (transpose, {'n_semitones': 12}),
    (transpose, {'n_semitones': 24}),
    (transpose, {'n_semitones': 46}),
    (transpose, {'n_semitones': -12}),
    (transpose, {'n_semitones': -24}),
    (transpose, {'n_semitones': -46}),
    #(split_duration, {'smallest_unit': 0.25}),
]

In [4]:
model_name="XGBoost"
model_name="RandomForest16"


In [5]:
# filein='midis/symphony_7_1_orch.mid'
fileout='midis/Autumn.mid'
#fileout='midis/fur-elise.mid'

# Multiclass, with transformations
amo_with_doublings_multiclass(filein=None,fileout=fileout,ytarget="instrument-name", model=model_name, tol=0.2, transformations=transformations, pipeline_path=f"weights/train_all_multiclass_transformations_{model_name}.joblib")

No input files for training.

Processing target file: midis/Autumn.mid
Number of events in midis/Autumn.mid : 4915
Last onset at 841.75
Original ticks_per_beat: 256

Transformation classification (training on orchestral file, prediction for target piano file)
Predicting transpose_{'n_semitones': 12} on target dataframe...
Prediction of transformation done. 4915 predictions generated.
Predicting transpose_{'n_semitones': 24} on target dataframe...
Prediction of transformation done. 4915 predictions generated.
Predicting transpose_{'n_semitones': 46} on target dataframe...
Prediction of transformation done. 4915 predictions generated.
Predicting transpose_{'n_semitones': -12} on target dataframe...
Prediction of transformation done. 4915 predictions generated.
Predicting transpose_{'n_semitones': -24} on target dataframe...
Prediction of transformation done. 4915 predictions generated.
Predicting transpose_{'n_semitones': -46} on target dataframe...
Prediction of transformation done. 491

{}

In [6]:
# Single class, without transformations
amo_with_doublings_multiclass(filein=None,fileout=fileout,ytarget="instrument-name", model=model_name, tol=0.2, transformations=None, multiclass=False, pipeline_path=f"weights/train_all_singleclass_notransformations_{model_name}.joblib")

No input files for training.

Processing target file: midis/Autumn.mid
Number of events in midis/Autumn.mid : 4915
Last onset at 841.75
Original ticks_per_beat: 256

Summary of target piano file (no transformations)
Number of events in midis/Autumn.mid : 4915
Last onset at 841.75

========= PREDICTION (target file) =========
Predictions  [ 4  6  7  8  9 11 13 14 15 17 20 22 23 27 28 30 33 36 38 39 40 41 42]
Predictions map ['Bassoon' 'Celesta' 'Clarinet' 'Contrabass' 'Contrabass and Violoncello'
 'Contrabassoon' 'Cymbal' 'Drum' 'English-Horn' 'Flute' 'Harp' 'Horn'
 'Oboe' 'Piano' 'Piccolo' 'Timpani' 'Triangle' 'Trumpet' 'Viola' 'Violin'
 'Violin and Viola and Violoncello and Contrabass' 'Violoncello' 'Voice']

========= POSTPROCESSING AND SAVING =========
Orchestration with preserved structure: midis/Autumn_RandomForest16_WITH_TIMING_TRAIN_ALL.mid

=== SAVING WITH EXACT TIMING STRUCTURE ===
Extracting timing structure from midis/Autumn.mid
ticks_per_beat: 256
Found 69 timing events:
  

{}